# 面试问题：Self-Consistency 多数投票何时有效，相关采样为什么会制造虚假多数？

        ## 可直接复述的回答主线

        1. Self-Consistency 通过采样多条推理路径并对最终答案投票，降低单条链路的偶然错误。
2. 简单多数投票隐含路径独立假设，同一模板复制出的相关错误会被重复计票。
3. 工程实现要先做稳定答案解析，再记录路径来源、采样簇、置信度和答案。
4. 相关性感知投票可让每个采样簇的总权重受限，使独立证据比重复模板更有价值。
5. 结果要同时展示准确率、胜出边际和最大簇占比，并在边际太小时选择 abstain。
6. 生产系统还需使用真实模型采样、语义答案判等、动态停止和成本预算。

        后续实验会用同一批输入依次验证朴素方案、核心机制、失败边界和修正效果。

## 1. 真实案例与输入预览

案例包含五个客服与运营推理问题，每题有五条离线推理路径、最终答案、采样模板簇和置信度。第一题故意包含三条同模板错误路径以复现相关采样压倒两条独立正确路径；文本为脱敏教学样本。

In [1]:
import re  # 规范化货币、标点和答案前缀以便稳定投票。
from collections import Counter, defaultdict  # 统计答案票数、采样簇大小和加权得分。
questions = [{"id": "vote-01", "question": "24箱每箱18件，共多少件？", "truth": "432", "paths": [{"answer": "420", "cluster": "template-A", "confidence": 0.78}, {"answer": "420", "cluster": "template-A", "confidence": 0.76}, {"answer": "420", "cluster": "template-A", "confidence": 0.74}, {"answer": "432", "cluster": "independent-B", "confidence": 0.90}, {"answer": "432", "cluster": "independent-C", "confidence": 0.86}]}, {"id": "vote-02", "question": "订单1260元打九折，应收多少元？", "truth": "1134", "paths": [{"answer": "1134", "cluster": "algebra-A", "confidence": 0.91}, {"answer": "1134", "cluster": "check-B", "confidence": 0.88}, {"answer": "126", "cluster": "template-C", "confidence": 0.40}, {"answer": "1134", "cluster": "algebra-A", "confidence": 0.84}, {"answer": "1124", "cluster": "guess-D", "confidence": 0.35}]}, {"id": "vote-03", "question": "退款3笔每笔80元，总额多少？", "truth": "240", "paths": [{"answer": "240", "cluster": "sum-A", "confidence": 0.90}, {"answer": "240", "cluster": "check-B", "confidence": 0.87}, {"answer": "160", "cluster": "omit-C", "confidence": 0.45}, {"answer": "240", "cluster": "sum-A", "confidence": 0.82}, {"answer": "320", "cluster": "extra-D", "confidence": 0.30}]}, {"id": "vote-04", "question": "延迟从800ms降到600ms，下降多少ms？", "truth": "200", "paths": [{"answer": "200", "cluster": "subtract-A", "confidence": 0.93}, {"answer": "200", "cluster": "check-B", "confidence": 0.89}, {"answer": "1400", "cluster": "add-C", "confidence": 0.22}, {"answer": "200", "cluster": "subtract-A", "confidence": 0.81}, {"answer": "20", "cluster": "unit-D", "confidence": 0.42}]}, {"id": "vote-05", "question": "5台服务每台12并发，总并发多少？", "truth": "60", "paths": [{"answer": "60", "cluster": "multiply-A", "confidence": 0.91}, {"answer": "60", "cluster": "check-B", "confidence": 0.88}, {"answer": "17", "cluster": "add-C", "confidence": 0.30}, {"answer": "60", "cluster": "multiply-A", "confidence": 0.84}, {"answer": "72", "cluster": "extra-D", "confidence": 0.28}]}]  # 定义五题及每题五条带相关簇的推理路径。
print("教学实验输入：五个问题，每题五条采样路径")  # 标记下方为离线相关采样案例。
for question in questions:  # 逐题展示真值和采样答案来源。
    compact_paths = [(path["answer"], path["cluster"], path["confidence"]) for path in question["paths"]]  # 整理当前题的可读路径摘要。
    print(f"{question['id']} truth={question['truth']} | {question['question']}")  # 输出当前问题和评测真值。
    print("  paths=", compact_paths)  # 输出答案、相关簇和置信度。

教学实验输入：五个问题，每题五条采样路径
vote-01 truth=432 | 24箱每箱18件，共多少件？
  paths= [('420', 'template-A', 0.78), ('420', 'template-A', 0.76), ('420', 'template-A', 0.74), ('432', 'independent-B', 0.9), ('432', 'independent-C', 0.86)]
vote-02 truth=1134 | 订单1260元打九折，应收多少元？
  paths= [('1134', 'algebra-A', 0.91), ('1134', 'check-B', 0.88), ('126', 'template-C', 0.4), ('1134', 'algebra-A', 0.84), ('1124', 'guess-D', 0.35)]
vote-03 truth=240 | 退款3笔每笔80元，总额多少？
  paths= [('240', 'sum-A', 0.9), ('240', 'check-B', 0.87), ('160', 'omit-C', 0.45), ('240', 'sum-A', 0.82), ('320', 'extra-D', 0.3)]
vote-04 truth=200 | 延迟从800ms降到600ms，下降多少ms？
  paths= [('200', 'subtract-A', 0.93), ('200', 'check-B', 0.89), ('1400', 'add-C', 0.22), ('200', 'subtract-A', 0.81), ('20', 'unit-D', 0.42)]
vote-05 truth=60 | 5台服务每台12并发，总并发多少？
  paths= [('60', 'multiply-A', 0.91), ('60', 'check-B', 0.88), ('17', 'add-C', 0.3), ('60', 'multiply-A', 0.84), ('72', 'extra-D', 0.28)]


## 2. Baseline / 基线：每条路径一票的简单多数

简单多数不区分路径是否来自同一模板。第一题三条 template-A 错误路径会形成 3:2 的虚假多数，即使两条正确路径更独立且置信度更高。

In [2]:
def majority_vote(paths):  # 对每条采样路径等权计票。
    counts = Counter(path["answer"] for path in paths)  # 汇总每个最终答案的路径数。
    winner = max(counts, key=lambda answer: (counts[answer], answer))  # 按票数和稳定字典序选出答案。
    return winner, counts  # 返回胜出答案和完整票数。
baseline_results = []  # 保存五题简单多数结果。
for question in questions:  # 逐题执行等权投票。
    winner, counts = majority_vote(question["paths"])  # 计算当前题多数答案。
    baseline_results.append({"id": question["id"], "winner": winner, "counts": counts, "correct": winner == question["truth"]})  # 保存投票分布和正确性。
print("Baseline 简单多数结果")  # 标记下表没有处理路径相关性。
print("问题      票数分布                  winner  correct")  # 输出逐题投票表头。
for result in baseline_results:  # 逐题展示答案票数和正确性。
    print(f"{result['id']:<9} {str(dict(result['counts'])):<25} {result['winner']:>7} {str(result['correct']):>8}")  # 输出当前题简单多数结果。

Baseline 简单多数结果
问题      票数分布                  winner  correct
vote-01   {'420': 3, '432': 2}          420    False
vote-02   {'1134': 3, '126': 1, '1124': 1}    1134     True
vote-03   {'240': 3, '160': 1, '320': 1}     240     True
vote-04   {'200': 3, '1400': 1, '20': 1}     200     True
vote-05   {'60': 3, '17': 1, '72': 1}      60     True


## 3. 底层实现：按采样簇分摊权重

同一 cluster 内每条路径权重为 confidence 除以簇大小，使复制同模板不会线性增加总票权；不同 cluster 的独立证据仍可累加。

In [3]:
def correlated_vote(paths):  # 对相关采样簇实施总权重受限的投票。
    cluster_sizes = Counter(path["cluster"] for path in paths)  # 统计每个模板簇复制了多少条路径。
    answer_scores = defaultdict(float)  # 累积每个答案的相关性修正得分。
    ledger = []  # 保存每条路径的原置信度和实际票权。
    for path in paths:  # 逐路径计算簇内分摊权重。
        weight = path["confidence"] / cluster_sizes[path["cluster"]]  # 把簇总影响限制在平均置信度量级。
        answer_scores[path["answer"]] += weight  # 把当前独立性修正权重计入答案。
        ledger.append({"answer": path["answer"], "cluster": path["cluster"], "confidence": path["confidence"], "weight": weight})  # 保存路径级投票过程。
    ordered = sorted(answer_scores.items(), key=lambda item: (-item[1], item[0]))  # 按得分降序和答案字典序稳定排序。
    winner = ordered[0][0]  # 读取最高相关性修正得分的答案。
    margin = ordered[0][1] - ordered[1][1] if len(ordered) > 1 else ordered[0][1]  # 计算胜出答案相对第二名的边际。
    return winner, dict(answer_scores), ledger, margin  # 返回答案、得分、路径账本和边际。
corrected_results = []  # 保存五题相关性感知投票结果。
for question in questions:  # 逐题执行簇权重修正。
    winner, scores_by_answer, ledger, margin = correlated_vote(question["paths"])  # 计算当前题的独立性修正结果。
    corrected_results.append({"id": question["id"], "winner": winner, "scores": scores_by_answer, "ledger": ledger, "margin": margin, "correct": winner == question["truth"]})  # 保存完整结果。
print("vote-01 路径级权重账本")  # 选择虚假多数题展示修正机制。
print("answer  cluster         confidence  weight")  # 输出路径账本表头。
for item in corrected_results[0]["ledger"]:  # 逐条展示第一题五条路径。
    print(f"{item['answer']:<7} {item['cluster']:<15} {item['confidence']:>10.2f} {item['weight']:>7.3f}")  # 输出当前路径分摊后的实际票权。
print("修正得分=", {answer: round(score, 3) for answer, score in corrected_results[0]["scores"].items()})  # 展示第一题答案总得分。

vote-01 路径级权重账本
answer  cluster         confidence  weight
420     template-A            0.78   0.260
420     template-A            0.76   0.253
420     template-A            0.74   0.247
432     independent-B         0.90   0.900
432     independent-C         0.86   0.860
修正得分= {'420': 0.76, '432': 1.76}


## 4. 结果表与结果解读

对照使用相同五题和相同最终答案。第一题的普通多数错误，簇修正后两条独立正确路径胜出；其余题保持正确，不是靠改评测集得到收益。

In [4]:
baseline_by_id = {result["id"]: result for result in baseline_results}  # 建立简单多数结果索引。
corrected_by_id = {result["id"]: result for result in corrected_results}  # 建立相关性修正结果索引。
baseline_accuracy = sum(result["correct"] for result in baseline_results) / len(questions)  # 计算简单多数五题准确率。
corrected_accuracy = sum(result["correct"] for result in corrected_results) / len(questions)  # 计算相关性感知投票准确率。
print("问题      truth  多数答案  修正答案  margin  变化")  # 输出同题结果对照表头。
for question in questions:  # 逐题比较两种投票策略。
    baseline_answer = baseline_by_id[question["id"]]["winner"]  # 读取简单多数答案。
    corrected = corrected_by_id[question["id"]]  # 读取修正答案和边际。
    change = "修正错误" if baseline_answer != question["truth"] and corrected["correct"] else "保持"  # 解释当前题是否发生有效变化。
    print(f"{question['id']:<9} {question['truth']:>5} {baseline_answer:>8} {corrected['winner']:>8} {corrected['margin']:>7.3f} {change:>8}")  # 输出当前题对照。
print(f"结果解读：简单多数准确率={baseline_accuracy:.1%}，相关性修正准确率={corrected_accuracy:.1%}；教学集很小，不能外推泛化收益。")  # 明确结果与外推边界。

问题      truth  多数答案  修正答案  margin  变化
vote-01     432      420      432   1.000     修正错误
vote-02    1134     1134     1134   1.355       保持
vote-03     240      240      240   1.280       保持
vote-04     200      200      200   1.340       保持
vote-05      60       60       60   1.455       保持
结果解读：简单多数准确率=80.0%，相关性修正准确率=100.0%；教学集很小，不能外推泛化收益。


## 5. 失败案例与修正

答案解析不稳定也会拆票。例如同一金额可写成“¥1,260”“1260元”“答案:1260”。先展示原始字符串被分成四类，再用受控数字规范化合并。

In [5]:
raw_answers = ["¥1,260", "1260元", "答案:1260", "1,260"]  # 构造四种语义相同但格式不同的金额答案。
naive_counts = Counter(raw_answers)  # 模拟直接按原始字符串投票的错误解析。
def normalize_numeric_answer(answer):  # 把金额格式规范化为纯数字答案。
    digits = re.sub(r"[^0-9-]", "", answer)  # 去除货币符号、千分位、中文和答案前缀。
    return str(int(digits)) if digits else "PARSE_ERROR"  # 返回规范整数并显式标记无数字输入。
normalized_answers = [normalize_numeric_answer(answer) for answer in raw_answers]  # 对四种格式执行同一规范化。
fixed_counts = Counter(normalized_answers)  # 对规范答案重新投票。
print("错误行为：原始字符串票数=", dict(naive_counts))  # 展示同义答案被拆成四票。
print("修正行为：规范化票数=", dict(fixed_counts))  # 展示稳定解析后的四票合并。

错误行为：原始字符串票数= {'¥1,260': 1, '1260元': 1, '答案:1260': 1, '1,260': 1}
修正行为：规范化票数= {'1260': 4}


## 6. 生产边界

cluster 标签在真实系统中需要由采样 seed、prompt 模板、解码轨迹或语义嵌入估计。还需处理开放式答案等价、模型共模错误、动态采样停止、成本预算和低边际 abstain。

In [6]:
first_cluster_sizes = Counter(path["cluster"] for path in questions[0]["paths"])  # 统计虚假多数题的最大相关簇。
max_cluster_share = max(first_cluster_sizes.values()) / len(questions[0]["paths"])  # 计算最大簇占全部路径比例。
abstain_threshold = 0.20  # 设定教学用低边际拒答阈值。
abstained = [result["id"] for result in corrected_results if result["margin"] < abstain_threshold]  # 找出证据边际不足的问题。
print(f"生产诊断：vote-01最大簇占比={max_cluster_share:.1%}，低边际需复核={abstained}")  # 输出相关性和拒答监控。

生产诊断：vote-01最大簇占比=60.0%，低边际需复核=[]


## 7. 最小回归测试

只验证样本规模、虚假多数复现、修正准确率和解析合并。

In [7]:
assert len(questions) >= 5  # 保证案例至少包含五个真实语义问题。
assert baseline_by_id["vote-01"]["winner"] != questions[0]["truth"]  # 保证第一题真实复现相关错误虚假多数。
assert corrected_by_id["vote-01"]["winner"] == questions[0]["truth"]  # 保证簇权重修正第一题答案。
assert corrected_accuracy > baseline_accuracy  # 保证同一教学集上的对照方向正确。
assert fixed_counts == Counter({"1260": 4})  # 保证四种金额格式被规范为同一答案。